# EXP-001 · Una neurona aprende la puerta lógica AND

**Laboratorio de IA | TensorFlow + Google Colab**  
**Nivel:** inicial · **Hardware:** CPU · **Tiempo de práctica:** 15–25 minutos

## Objetivo

Comprender, con el modelo más pequeño posible, la diferencia entre **datos de entrenamiento**, **modelo**, **entrenamiento** y **predicción**.

Una puerta lógica **AND** da `1` únicamente cuando **las dos entradas valen 1**.

| Entrada 1 | Entrada 2 | Salida AND |
|:--:|:--:|:--:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

**Pregunta antes de comenzar:** ¿qué salida debería producir la entrada `[1, 0]`?

**Material complementario:** [explicación detallada en Markdown](001_perceptron_and_explicacion_detallada.md).

**Cómo utilizarlo:** ejecuta las celdas de arriba abajo. Lee los comentarios `#` del código: indican qué hace cada línea y qué significa cada parámetro.

## 1. Importar las herramientas

- `numpy` permite representar los datos como matrices.
- `tensorflow` construye y entrena la red neuronal.
- `matplotlib.pyplot` dibuja el progreso del aprendizaje.

No necesitas una TPU: con solo cuatro ejemplos y una neurona, la CPU es más que suficiente. En Colab puedes elegir **Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU**.

Si Colab indica que TensorFlow no está instalado, ejecuta en otra celda `%pip install tensorflow` y reinicia el entorno.

In [ ]:
# NumPy: nos ayuda a manejar matrices (colecciones de números).
import numpy as np

# TensorFlow y su interfaz Keras: crean, entrenan y utilizan redes neuronales.
import tensorflow as tf

# Matplotlib: representamos el error y los aciertos con gráficas.
import matplotlib.pyplot as plt

# Fijamos una semilla aleatoria para facilitar la reproducción de la prueba.
# No garantiza resultados idénticos en todos los equipos y versiones.
tf.keras.utils.set_random_seed(7)

# Comprobamos qué versión de TensorFlow está disponible en Google Colab.
print("TensorFlow:", tf.__version__)

# Mostramos los dispositivos CPU que reconoce TensorFlow.
# Para solo cuatro ejemplos, no hace falta habilitar una GPU ni una TPU.
print("Dispositivos CPU:", tf.config.list_physical_devices("CPU"))

## 2. Preparar los datos

Cada fila de `X` representa **una muestra de entrenamiento**, formada por **dos entradas**. Cada elemento de `y` es la respuesta correcta para esa muestra.

- `X.shape` será `(4, 2)`: 4 ejemplos × 2 características.
- `y.shape` será `(4, 1)`: 4 respuestas × 1 salida.

**No hay imágenes todavía:** este experimento enseña el funcionamiento básico de una neurona que después podremos aplicar a píxeles.

In [ ]:
# X contiene la tabla de verdad AND: cuatro filas, una por ejemplo.
# Cada fila tiene DOS entradas: [x1, x2].
X = np.array([
    [0, 0],  # Caso 1: 0 AND 0 = 0
    [0, 1],  # Caso 2: 0 AND 1 = 0
    [1, 0],  # Caso 3: 1 AND 0 = 0
    [1, 1]   # Caso 4: 1 AND 1 = 1
], dtype=np.float32)  # float32 es un tipo numérico habitual en TensorFlow.

# y contiene las respuestas correctas ("etiquetas"), en el MISMO orden que X.
# Cada respuesta se escribe como [0] o [1] para obtener forma (4, 1).
y = np.array([[0], [0], [0], [1]], dtype=np.float32)

# .shape indica las dimensiones: (4, 2) para X y (4, 1) para y.
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

# zip(X, y) recorre simultáneamente cada entrada y su salida correcta.
for entradas, salida in zip(X, y):
    # astype(int) e int(...) muestran 0/1 en vez de 0.0/1.0.
    print(f"Entradas: {entradas.astype(int)}  ->  Salida esperada: {int(salida[0])}")

## 3. Crear el modelo: **una sola neurona**

La neurona calcula una combinación de sus dos entradas, les añade un **sesgo** y transforma el resultado mediante una función **sigmoide**:

\[
 p = \sigma(w_1 x_1 + w_2 x_2 + b)
\]

- `w1` y `w2`: pesos que el entrenamiento ajusta.
- `b`: sesgo que también aprende.
- `sigmoid`: convierte el resultado en un número entre 0 y 1.

Este modelo es una neurona de salida sigmoide (un clasificador lineal); no es exactamente el **perceptrón clásico con función escalón**, aunque permite estudiar una idea muy similar.

In [ ]:
# Sequential indica que las capas se ejecutan en orden.
modelo = tf.keras.Sequential([
    # Input(shape=(2,)): cada ejemplo de X tiene dos características.
    # El 2 NO significa que tengamos dos ejemplos de entrenamiento.
    tf.keras.Input(shape=(2,)),

    # Dense(1): UNA neurona que recibe las dos entradas.
    # Aprende 2 pesos (w1, w2) y 1 sesgo (b): tres parámetros.
    # sigmoid transforma la suma ponderada en un número entre 0 y 1.
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Muestra capas, tamaño de salida y número de parámetros entrenables.
# La arquitectura está creada, pero todavía NO se ha entrenado.
modelo.summary()

## 4. Configurar el aprendizaje: `compile()`

- **`optimizer="adam"`**: método para modificar los pesos y reducir los errores.
- **`loss="binary_crossentropy"`**: mide el error de una clasificación binaria (`0` o `1`) en términos de probabilidades.
- **`metrics=["accuracy"]`**: calcula el porcentaje de aciertos tras aplicar el umbral de clasificación.

Usamos una tasa de aprendizaje explícita de `0.1` para que esta miniatura aprenda rápidamente. Para tareas grandes no tiene por qué ser la mejor elección.

In [ ]:
# compile() CONFIGURA cómo aprenderá el modelo, pero NO lo entrena.
modelo.compile(
    # Adam modifica los pesos y el sesgo a partir de los gradientes.
    # learning_rate=0.1 controla el tamaño aproximado de las actualizaciones.
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),

    # Pérdida apropiada cuando la respuesta correcta es 0 o 1
    # y la salida de la neurona es una probabilidad estimada.
    loss="binary_crossentropy",

    # Accuracy: fracción de ejemplos clasificados correctamente.
    # Para calcular la clase de esta salida sigmoide, se utiliza umbral 0.5.
    metrics=["accuracy"]
)

## 5. Entrenar: `fit()`

En cada **época** el modelo utiliza las cuatro muestras para corregir sus parámetros. El método `fit()` **sí modifica los pesos**. Guardamos el historial para analizar lo que sucedió.

- `epochs=300`: 300 pasadas por el conjunto completo.
- `verbose=0`: no muestra una línea por época.
- `history.history`: contiene el historial de la pérdida y la precisión.

El resultado exacto puede variar si cambias la semilla, las épocas o la tasa de aprendizaje.

In [ ]:
# fit() SÍ entrena: modifica los pesos y el sesgo para reducir el error.
history = modelo.fit(
    X,          # Entradas de los cuatro ejemplos.
    y,          # Respuestas correctas asociadas a las entradas.
    epochs=300, # 300 pasadas completas por estos cuatro ejemplos.
    verbose=0   # No mostrar el progreso de cada época en pantalla.
)

# history guarda los valores calculados a lo largo del entrenamiento.
# El índice [-1] extrae el ÚLTIMO valor registrado.
perdida_final = history.history["loss"][-1]
precision_final = history.history["accuracy"][-1]

# Loss cuantifica el error de las probabilidades.
print("Pérdida final:", round(perdida_final, 5))

# Accuracy es una proporción de 0 a 1; multiplicamos por 100 para expresarla en %.
print("Precisión de entrenamiento:", round(100 * precision_final, 1), "%")

# Importante: si vuelves a ejecutar esta celda sin recrear el modelo,
# fit() continúa entrenando desde los pesos actuales, no desde cero.

## 6. Ver cómo ha aprendido

Si el entrenamiento funciona, la **pérdida** (error) suele bajar. La **precisión** suele subir y puede alcanzar el 100 % en estos cuatro ejemplos.

⚠️ **Importante:** estamos evaluando los mismos cuatro casos con los que se ha entrenado. Este resultado **no demuestra** que el modelo generalice bien a datos distintos.

In [ ]:
# --- Gráfica 1: evolución de la pérdida (loss) ---
plt.figure(figsize=(8, 4))             # Tamaño de la figura en pulgadas.
plt.plot(history.history["loss"], label="Error (loss)")
plt.xlabel("Época")                    # Eje horizontal: número de época.
plt.ylabel("Pérdida")                  # Eje vertical: error de entrenamiento.
plt.title("Cómo disminuye el error durante el entrenamiento")
plt.grid(True)                         # Rejilla para facilitar la lectura.
plt.legend()                           # Muestra la etiqueta de la curva.
plt.show()                             # Dibuja la figura.

# --- Gráfica 2: evolución de los aciertos (accuracy) ---
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Precisión (accuracy)")
plt.xlabel("Época")
plt.ylabel("Precisión")
plt.ylim(0, 1.05)                       # El 100 % equivale a 1.0.
plt.title("Porcentaje de aciertos durante el entrenamiento")
plt.grid(True)
plt.legend()
plt.show()

# La pérdida puede seguir bajando aunque la precisión ya sea del 100 %:
# acertar las clases no significa que las probabilidades sean perfectas.

## 7. Usar el modelo entrenado: `predict()`

Esta es la parte de **inferencia**: el modelo utiliza los pesos que ya ha aprendido para calcular respuestas. **`predict()` no entrena ni modifica los pesos.**

La función sigmoide devuelve un número entre 0 y 1. Decidimos que:

- `p >= 0.5` → salida `1`.
- `p < 0.5` → salida `0`.

In [ ]:
# predict() realiza INFERENCIA: usa los pesos aprendidos sin entrenar.
# Devuelve cuatro probabilidades, una para cada fila de X.
probabilidades = modelo.predict(X, verbose=0)

print("Entradas | Probabilidad de 1 | Predicción | Resultado correcto")
# zip permite recorrer los cuatro ejemplos, sus probabilidades y sus etiquetas.
for entradas, prob, esperado in zip(X, probabilidades, y):
    # prob[0] es la salida numérica de nuestra única neurona.
    # Aplicamos el umbral 0.5 para convertir probabilidad en clase 0 o 1.
    clase = 1 if prob[0] >= 0.5 else 0

    # :.4f limita la visualización de la probabilidad a cuatro decimales.
    print(f"{entradas.astype(int)}     | {prob[0]:.4f}            | {clase}          | {int(esperado[0])}")

# --- Predicción individual ---
# Los dos corchetes son importantes: (1, 2) significa
# una muestra con dos entradas, no dos muestras.
nueva_entrada = np.array([[1, 0]], dtype=np.float32)

# [0, 0] elige la primera muestra y su primera (única) salida.
p = float(modelo.predict(nueva_entrada, verbose=0)[0, 0])

# int(p >= 0.5) transforma True/False en 1/0.
print(f"\nPara [1, 0], el modelo devuelve {p:.4f}; clase = {int(p >= 0.5)}")

## 8. Inspeccionar lo que realmente aprendió

No ha memorizado una lista de instrucciones como un programa tradicional: la neurona ha ajustado **dos pesos y un sesgo**. Podemos ver esos valores.

Si el modelo ha aprendido AND, el sesgo suele hacer que se necesiten las dos entradas activas para alcanzar una probabilidad alta.

In [ ]:
# get_weights() devuelve parámetros aprendidos de la primera capa Dense.
# pesos tiene forma (2, 1): dos entradas por una neurona.
# sesgo contiene un único valor: b.
pesos, sesgo = modelo.layers[0].get_weights()

print(f"Peso de entrada 1 (w1): {pesos[0, 0]:.4f}")
print(f"Peso de entrada 2 (w2): {pesos[1, 0]:.4f}")
print(f"Sesgo (b):              {sesgo[0]:.4f}")

# Verificamos manualmente la neurona para la entrada [1, 1].
# Primero calculamos la suma ponderada: z = 1*w1 + 1*w2 + b.
z = pesos[0, 0] * 1 + pesos[1, 0] * 1 + sesgo[0]

# Después aplicamos la misma función sigmoide que utiliza Dense.
# np.exp(-z) calcula e elevado a -z.
prob_manual = 1 / (1 + np.exp(-z))

print(f"Sigmoide calculada manualmente para [1, 1]: {prob_manual:.4f}")

# La salida debe estar muy cerca de modelo.predict([[1, 1]]).
# Hemos reproducido el cálculo sin utilizar predict().

## 9. Tres pequeñas pruebas para seguir aprendiendo

1. **Menos entrenamiento:** cambia `epochs=300` por `epochs=10`, vuelve a ejecutar desde la celda que crea el modelo y compara las predicciones. ¿Ha aprendido suficientemente?
2. **Aprender OR:** sustituye `y` por `[[0], [1], [1], [1]]` y ejecuta de nuevo desde la creación del modelo. ¿Puede aprenderlo una neurona?
3. **Intentar XOR:** sustituye `y` por `[[0], [1], [1], [0]]`. ¿Por qué una sola neurona sigmoide no puede clasificar correctamente los cuatro casos de XOR con un umbral lineal? Esta pregunta dará pie al siguiente experimento con una **capa oculta**.

> Siempre que cambies `y`, reconstruye también `modelo` y ejecuta de nuevo `compile()` y `fit()` para que la comparación parta de un modelo nuevo.

### Ficha de conclusiones (rellenar después de probar)

- **Qué esperaba:** ...
- **Qué ocurrió:** ...
- **Pérdida final:** ...
- **Precisión final (sobre los datos de entrenamiento):** ...
- **Qué hacen `compile()`, `fit()` y `predict()`:** ...
- **Siguiente experimento:** pasar de dos entradas a imágenes de píxeles o probar XOR con capas ocultas.

### Guardarlo en tu laboratorio de GitHub

Guarda este cuaderno como `01_fundamentos/001_perceptron_and.ipynb` y añade un enlace en el `README.md` de tu laboratorio. Desde Colab puedes usar **Archivo → Guardar una copia en GitHub** si has conectado tu cuenta.
